# Instacart Next-Basket Recommendation — Clean Rebuild

**XCAI2-inspired feature model · Instacart Market Basket Analysis · Google Colab**

Notebook này được xây dựng lại từ đầu theo hướng nghiên cứu đã thống nhất: dự đoán **một giỏ hàng kế tiếp** từ **H giỏ hàng gần nhất**, với `H = 4, 8, 13`.

- **Baseline:** User History Frequency, Recency-weighted History, Global Popularity.
- **Mô hình so sánh:** XCAI2-inspired user–product feature model với LightGBM.
- **Mô hình chính của đề tài:** Sets2Sets (được chạy bằng source riêng, dùng cùng split và item mapping xuất ra ở notebook này).

> Mặc định `MAX_USERS = 1000` chỉ là smoke test để kiểm tra pipeline. Chưa phải thực nghiệm cuối cùng của đề tài.

## Research protocol và kiểm soát leakage

| Thành phần | Quy ước |
|---|---|
| Input | H giỏ `prior` mới nhất của mỗi user |
| Target | Một order có `eval_set='train'` của user đó |
| H | 4, 8, 13 |
| Split | Theo user, 72% train / 8% validation / 20% test; cố định cho mọi H |
| Vocabulary | Top sản phẩm từ lịch sử prior của TRAIN users בלבד |
| Candidate | Sản phẩm trong H-history ∪ Top-N global popularity từ TRAIN H-history |
| Comparator | LightGBM xếp hạng candidate dựa trên feature user/product/category |
| Metrics | Precision@K, Recall@K, NDCG@K, HitRate@K, MAP@K, candidate coverage |

**Các điểm chống leakage:**

1. Chỉ dùng lịch sử `prior` làm input. Không dùng feature lấy từ target order (ví dụ `days_since_prior_order` của target).
2. Global popularity và product/aisle/department statistics chỉ tính từ H-history của TRAIN users.
3. Không dùng cột `reordered` gốc của Instacart; tính lại trạng thái mua lặp trong cửa sổ H.
4. Target basket chỉ dùng làm label. Các positive train được thêm vào candidate rows để tạo ví dụ supervised, nhưng không đưa target-derived flags vào feature model.
5. Validation/test candidate generation không nhìn target basket.
6. Recall/NDCG/MAP lấy toàn bộ mapped target basket làm mẫu số; candidate coverage được báo riêng.
7. Target-item coverage sau khi giới hạn vocabulary được báo riêng với candidate coverage.

**Giới hạn dataset:** `eval_set='test'` của cuộc thi Instacart không có product labels công khai. Vì vậy đây là offline evaluation trên nhóm `eval_set='train'`, sau đó chia user thành train/validation/test; không phải điểm số trên hidden test của Kaggle.

## Cell 1 — Cài các thư viện bổ sung

Colab đã cung cấp sẵn NumPy, SciPy, Pandas, Scikit-learn và Matplotlib. **Không cài đè hoặc force-reinstall các thư viện nền này** vì có thể làm lệch các binary extension trong runtime.

Cell này chỉ cài KaggleHub để tải dataset và LightGBM làm mô hình XCAI2-inspired.

In [ ]:
%pip install -q kagglehub lightgbm

print("KaggleHub and LightGBM installation finished.")

### Quan trọng — Restart session sau khi cài package bổ sung

Sau khi Cell 1 chạy xong, chọn **Runtime → Restart session**. Việc restart giúp Python nạp package mới nhất trong cùng runtime. Không cần reinstall NumPy/SciPy/Pandas/Scikit-learn. Sau restart, chạy tiếp cell kiểm tra môi trường.

## Cell 2 — Kiểm tra môi trường và LightGBM smoke test

Cell này xác nhận các thư viện nền của Colab import được, rồi fit một LightGBM rất nhỏ để kiểm tra khả năng huấn luyện trước khi đọc dataset lớn.

In [ ]:
import sys
import numpy as np
import pandas as pd
import scipy
import sklearn
import lightgbm
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)

# Small CPU-only fit: verifies LightGBM can train with the active NumPy stack.
X_check = np.array([[0.0, 1.0], [1.0, 0.0], [0.2, 0.8], [0.8, 0.2],
                    [0.1, 0.9], [0.9, 0.1], [0.3, 0.7], [0.7, 0.3]], dtype=np.float32)
y_check = np.array([0, 1, 0, 1, 0, 1, 0, 1], dtype=np.int8)
model_check = LGBMClassifier(n_estimators=5, max_depth=2, verbosity=-1, n_jobs=1, random_state=42)
model_check.fit(X_check, y_check)
pred_check = model_check.predict_proba(X_check)
assert pred_check.shape == (len(y_check), 2)
print("\nEnvironment and LightGBM smoke test: PASSED")

del X_check, y_check, model_check, pred_check

## Cell 3 — Kết nối Google Drive và tạo thư mục dự án

In [ ]:
from google.colab import drive
from pathlib import Path
import json, os, shutil, time

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Instacart_NBR_Clean')
DATA_ROOT = PROJECT_ROOT / 'data'
DATA_DIR = DATA_ROOT / 'instacart'
EXPERIMENTS_ROOT = PROJECT_ROOT / 'experiments'

for folder in [PROJECT_ROOT, DATA_ROOT, DATA_DIR, EXPERIMENTS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Experiments directory:", EXPERIMENTS_ROOT)

## Cell 4 — Tải dataset Instacart trực tiếp vào Drive

In [ ]:
import kagglehub

REQUIRED_FILES = [
    'orders.csv',
    'order_products__prior.csv',
    'order_products__train.csv',
    'products.csv',
    'aisles.csv',
    'departments.csv',
]

def find_dataset_folder(search_root):
    search_root = Path(search_root)
    if not search_root.exists():
        return None
    for orders_file in search_root.rglob('orders.csv'):
        folder = orders_file.parent
        if all((folder / name).is_file() for name in REQUIRED_FILES):
            return folder
    return None

found = find_dataset_folder(DATA_DIR)
if found is None:
    print("Dataset chưa có đủ 6 CSV. Bắt đầu tải từ KaggleHub...")
    download_path = kagglehub.dataset_download(
        'psparks/instacart-market-basket-analysis',
        output_dir=str(DATA_DIR)
    )
    print("KaggleHub returned:", download_path)
    found = find_dataset_folder(DATA_DIR) or find_dataset_folder(download_path)
else:
    print("Đã tìm thấy dataset có sẵn:", found)

if found is None:
    raise FileNotFoundError(
        "Không tìm thấy đủ 6 CSV sau khi tải. Hãy kiểm tra lỗi KaggleHub ở cell này."
    )

DATA_DIR = Path(found)
print("\nDataset ready at:", DATA_DIR)
for filename in REQUIRED_FILES:
    p = DATA_DIR / filename
    print(f"OK | {filename:32s} | {p.stat().st_size / 1024**2:8.1f} MiB")

## Cell 5 — Kiểm tra cấu trúc và thống kê dataset

In [ ]:
ORDER_USECOLS = [
    'order_id', 'user_id', 'eval_set', 'order_number',
    'order_dow', 'order_hour_of_day', 'days_since_prior_order'
]
ORDER_DTYPES = {
    'order_id': 'int32', 'user_id': 'int32', 'eval_set': 'category',
    'order_number': 'int16', 'order_dow': 'int8',
    'order_hour_of_day': 'int8', 'days_since_prior_order': 'float32'
}

orders_eda = pd.read_csv(
    DATA_DIR / 'orders.csv', usecols=ORDER_USECOLS, dtype=ORDER_DTYPES
)
print("Orders shape:", orders_eda.shape)
print("Unique users:", orders_eda['user_id'].nunique())
print("\nOrders by eval_set:")
display(orders_eda['eval_set'].value_counts(dropna=False).rename_axis('eval_set').to_frame('orders'))

prior_counts_eda = orders_eda.loc[orders_eda['eval_set'].astype(str).eq('prior')].groupby('user_id')['order_id'].nunique()
train_users_eda = orders_eda.loc[orders_eda['eval_set'].astype(str).eq('train'), 'user_id'].nunique()
print(f"Users with labeled eval_set='train' target: {train_users_eda:,}")
print(f"Labeled users having >=13 prior orders: {(prior_counts_eda >= 13).sum():,}")
print("\nFive sample rows:")
display(orders_eda.head())
del orders_eda, prior_counts_eda

## Cell 6 — Cấu hình thực nghiệm

**Smoke test trước:** giữ `MAX_USERS = 1000`. Khi đã chạy hết H=4/8/13 và kiểm tra kết quả, tạo `RUN_TAG` mới rồi nâng lên 5.000 hoặc toàn bộ user đủ điều kiện.

Không tái sử dụng cùng một thư mục run khi thay đổi `MAX_USERS`, `TOP_ITEMS` hoặc `SEED`.

In [ ]:
HISTORY_LENGTHS = [4, 8, 13]
MAX_USERS = 1000          # Smoke test. Use 5000 or None for larger experiments.
TOP_ITEMS = 6000          # Shared item vocabulary from TRAIN users' prior histories.
GLOBAL_TOP_N = 300        # Number of global candidates per user.
K_LIST = [5, 10, 20]
SEED = 42
CHUNK_SIZE = 500_000

RUN_TAG = f"smoke_users_{MAX_USERS}_items_{TOP_ITEMS}_seed_{SEED}"
RUN_DIR = EXPERIMENTS_ROOT / RUN_TAG
PREPARED_DIR = RUN_DIR / 'prepared'
RESULTS_ROOT = RUN_DIR / 'results'

for folder in [RUN_DIR, PREPARED_DIR, RESULTS_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

print("HISTORY_LENGTHS:", HISTORY_LENGTHS)
print("MAX_USERS:", MAX_USERS)
print("TOP_ITEMS:", TOP_ITEMS)
print("GLOBAL_TOP_N:", GLOBAL_TOP_N)
print("RUN_DIR:", RUN_DIR)

## Cell 7 — Tạo thư mục source trong runtime Colab

Cell kế tiếp sẽ ghi module pipeline đầy đủ vào thư mục này. Source cũng được đóng gói riêng trong ZIP đi kèm notebook.

In [ ]:
LOCAL_PROJECT = Path('/content/instacart_nbr_clean')
LOCAL_SRC = LOCAL_PROJECT / 'src'
LOCAL_SRC.mkdir(parents=True, exist_ok=True)
print("Local source directory:", LOCAL_SRC)

## Cell 8 — Ghi source code pipeline đầy đủ

In [ ]:
%%writefile /content/instacart_nbr_clean/src/nbr_pipeline.py
"""Instacart Next-Basket Recommendation: XCAI2-inspired feature model.

This implementation adapts the user-product candidate-scoring idea from
xcai2/Instacart-Market-Basket-Analysis to a controlled next-basket experiment.

Protocol
--------
* One labeled target basket per user: orders.eval_set == 'train'.
* Input: only the latest H prior baskets, H in {4, 8, 13}.
* User-level split: train/validation/test = 72/8/20, fixed across H.
* Candidate retrieval at validation/test time: personal H-history items UNION
  top global products computed only from TRAIN users' H-history.
* Train positives are included as supervised positive examples, but no
  target-derived feature or candidate-source flag is exposed to the model.
* Raw Instacart `reordered` is not used; within-window repeats are recomputed.
* Target items outside the shared item vocabulary are reported separately.

The model is a feature-based comparison to Sets2Sets, not a new architecture.
"""
from __future__ import annotations

import json
import math
import time
from collections import Counter
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

REQUIRED_FILES = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]

ORDER_COLUMNS = [
    "order_id", "user_id", "eval_set", "order_number", "order_dow",
    "order_hour_of_day", "days_since_prior_order",
]
ORDER_DTYPES = {
    "order_id": "int32", "user_id": "int32", "eval_set": "category",
    "order_number": "int16", "order_dow": "int8",
    "order_hour_of_day": "int8", "days_since_prior_order": "float32",
}
LINE_DTYPES = {
    "order_id": "int32", "product_id": "int32", "add_to_cart_order": "int16",
}

FEATURE_COLUMNS = [
    # User-window behaviour
    "u_hist_baskets", "u_hist_item_count", "u_unique_products",
    "u_avg_basket_size", "u_basket_size_std", "u_reorder_rate",
    "u_avg_days_gap", "u_std_days_gap", "u_dow_std", "u_hour_std",
    # User-product interaction inside H
    "up_purchase_count", "up_basket_frequency", "up_reorder_rate",
    "up_last_seen_age", "up_recency_score", "up_in_last_basket",
    "up_last3_count", "up_avg_cart_position", "up_recency_weighted_count",
    # Product stats calculated using TRAIN users' H-history only
    "p_total_purchases_train", "p_distinct_users_train",
    "p_reorder_rate_train", "p_avg_cart_position_train",
    "p_global_pop_score", "p_global_pop_rank", "p_is_organic",
    # Aisle and department statistics from TRAIN users' H-history only
    "a_total_purchases_train", "a_reorder_rate_train",
    "d_total_purchases_train", "d_reorder_rate_train",
    # User-category affinities calculated inside each user's H-history
    "ua_purchase_count", "ua_purchase_share",
    "ud_purchase_count", "ud_purchase_share",
]


def verify_dataset(data_dir: str | Path) -> Path:
    """Validate that the six expected CSV files exist in one directory."""
    root = Path(data_dir)
    missing = [name for name in REQUIRED_FILES if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Dataset folder is incomplete: {root}\nMissing files: {missing}"
        )
    return root


def _read_selected_lines(
    path: Path,
    wanted_order_ids: set[int],
    chunksize: int = 500_000,
) -> pd.DataFrame:
    """Read selected order-product rows from the large `prior` CSV in chunks."""
    if not wanted_order_ids:
        return pd.DataFrame({c: pd.Series(dtype=t) for c, t in LINE_DTYPES.items()})
    parts: list[pd.DataFrame] = []
    rows_kept = 0
    for chunk in pd.read_csv(
        path,
        usecols=list(LINE_DTYPES),
        dtype=LINE_DTYPES,
        chunksize=chunksize,
    ):
        part = chunk.loc[chunk["order_id"].isin(wanted_order_ids)]
        if not part.empty:
            parts.append(part)
            rows_kept += len(part)
    if not parts:
        return pd.DataFrame({c: pd.Series(dtype=t) for c, t in LINE_DTYPES.items()})
    result = pd.concat(parts, ignore_index=True)
    if result.duplicated(["order_id", "product_id"]).any():
        raise ValueError(f"Duplicate (order_id, product_id) rows detected in {path.name}.")
    print(f"  {path.name}: retained {rows_kept:,} selected item rows")
    return result


def _split_users(users: list[int], seed: int) -> dict[str, list[int]]:
    rng = np.random.default_rng(seed)
    shuffled = np.asarray(sorted(users), dtype=np.int64)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * 0.72)
    n_valid = int(n * 0.08)
    # Ensure that all three groups are non-empty for small smoke-test cohorts.
    if n >= 10:
        n_train = max(1, n_train)
        n_valid = max(1, n_valid)
        n_test = n - n_train - n_valid
        if n_test < 1:
            n_test = 1
            n_train -= 1
    else:
        raise ValueError("At least 10 eligible users are required for a 72/8/20 split.")
    return {
        "train": shuffled[:n_train].astype(int).tolist(),
        "valid": shuffled[n_train:n_train + n_valid].astype(int).tolist(),
        "test": shuffled[n_train + n_valid:].astype(int).tolist(),
    }


def _write_sets2sets_files(
    out_dir: Path,
    history_meta: pd.DataFrame,
    history_lines: pd.DataFrame,
    target_lines: pd.DataFrame,
    item_map: dict[int, int],
    users: list[int],
    history_lengths: list[int],
) -> None:
    """Export H-specific histories/targets in the companion Sets2Sets CSV format."""
    # Save into the shared prepared directory so the companion Sets2Sets runner
    # can consume the same user split, item mapping and H-specific CSVs directly.
    prepared = out_dir
    prepared.mkdir(parents=True, exist_ok=True)
    for h in history_lengths:
        meta_h = history_meta.sort_values(["user_id", "order_number"]).groupby(
            "user_id", sort=False, group_keys=False
        ).tail(h)
        h_ids = set(meta_h["order_id"].astype(int))
        h_lines = history_lines.loc[history_lines["order_id"].isin(h_ids)].copy()
        h_lines["model_product_id"] = h_lines["product_id"].map(item_map)
        future = target_lines.copy()
        future["model_product_id"] = future["product_id"].map(item_map)
        # The original format uses these names. Product IDs are remapped to a compact vocabulary.
        h_out = h_lines[["user_id", "order_number", "model_product_id"]].dropna().copy()
        f_out = future[["user_id", "target_order_number", "model_product_id"]].dropna().copy()
        h_out["model_product_id"] = h_out["model_product_id"].astype("int32")
        f_out["model_product_id"] = f_out["model_product_id"].astype("int32")
        h_out.columns = ["CUSTOMER_ID", "ORDER_NUMBER", "MATERIAL_NUMBER"]
        f_out.columns = ["CUSTOMER_ID", "ORDER_NUMBER", "MATERIAL_NUMBER"]
        h_out.to_csv(prepared / f"instacart_H{h}_history.csv", index=False)
        f_out.to_csv(prepared / f"instacart_H{h}_future.csv", index=False)


def prepare_cohort(
    data_dir: str | Path,
    out_dir: str | Path,
    history_lengths: Iterable[int] = (4, 8, 13),
    max_users: int | None = 1000,
    top_items: int = 6000,
    seed: int = 42,
    chunksize: int = 500_000,
) -> dict:
    """Create one fixed user split, shared item vocabulary and H-history cache.

    Only users with a labeled eval_set='train' target order and at least
    max(history_lengths) prior baskets are eligible. `max_users` is a smoke-test
    limit sampled after eligibility filtering; set None for all eligible users.
    """
    data_dir = verify_dataset(data_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    hs = sorted(set(int(h) for h in history_lengths))
    if not hs or min(hs) < 1:
        raise ValueError("history_lengths must be positive integers.")
    max_h = max(hs)
    if top_items < 200:
        raise ValueError("top_items should be at least 200 for a useful shared vocabulary.")

    print("Loading order metadata...")
    orders = pd.read_csv(
        data_dir / "orders.csv", usecols=ORDER_COLUMNS, dtype=ORDER_DTYPES
    )
    prior_orders_all = orders.loc[
        orders["eval_set"].astype(str).eq("prior"), ORDER_COLUMNS
    ].copy()
    target_orders_all = orders.loc[
        orders["eval_set"].astype(str).eq("train"), ["order_id", "user_id", "order_number"]
    ].copy()
    prior_counts = prior_orders_all.groupby("user_id")["order_id"].nunique()
    target_users = set(target_orders_all["user_id"].astype(int))
    eligible = sorted(
        int(uid) for uid in target_users
        if int(prior_counts.get(uid, 0)) >= max_h
    )
    if not eligible:
        raise ValueError(f"No labeled users have at least {max_h} prior baskets.")
    rng = np.random.default_rng(seed)
    rng.shuffle(eligible)
    if max_users is not None:
        if max_users < 10:
            raise ValueError("max_users must be >= 10 or None.")
        eligible = eligible[:min(int(max_users), len(eligible))]
    selected = set(eligible)
    splits = _split_users(eligible, seed)
    split_map = {uid: split for split, ids in splits.items() for uid in ids}

    prior_meta = prior_orders_all.loc[prior_orders_all["user_id"].isin(selected)].copy()
    prior_meta = prior_meta.sort_values(["user_id", "order_number"]).groupby(
        "user_id", sort=False, group_keys=False
    ).tail(max_h).copy()
    target_meta = target_orders_all.loc[target_orders_all["user_id"].isin(selected)].copy()
    target_meta = target_meta.rename(columns={"order_number": "target_order_number"})
    target_order_ids = set(target_meta["order_id"].astype(int))
    prior_order_ids = set(prior_meta["order_id"].astype(int))

    print(f"Eligible users selected: {len(selected):,}")
    print(f"Reading product rows for the latest {max_h} prior baskets per user...")
    prior_lines = _read_selected_lines(
        data_dir / "order_products__prior.csv", prior_order_ids, chunksize
    )
    # The labeled train file is much smaller; read it once and retain selected target orders.
    target_lines = pd.read_csv(
        data_dir / "order_products__train.csv",
        usecols=list(LINE_DTYPES), dtype=LINE_DTYPES,
    )
    target_lines = target_lines.loc[target_lines["order_id"].isin(target_order_ids)].copy()
    if prior_lines.empty or target_lines.empty:
        raise ValueError("No selected order-product rows were found; verify Instacart files.")

    prior_lines = prior_lines.merge(
        prior_meta[ORDER_COLUMNS], on="order_id", how="inner", validate="many_to_one"
    )
    target_lines = target_lines.merge(
        target_meta, on="order_id", how="inner", validate="many_to_one"
    )

    print("Building shared product vocabulary from TRAIN users' prior history only...")
    train_users_initial = set(splits["train"])
    vocab_counts = prior_lines.loc[
        prior_lines["user_id"].isin(train_users_initial), "product_id"
    ].value_counts()
    vocab_original = vocab_counts.head(top_items).index.astype(int).tolist()
    if len(vocab_original) < 100:
        raise ValueError("Shared product vocabulary has fewer than 100 products.")
    item_map = {int(old): i + 1 for i, old in enumerate(vocab_original)}
    allowed = set(item_map)

    # Load product dimensions, then keep original Instacart product IDs for model features.
    dims = pd.read_csv(
        data_dir / "products.csv",
        usecols=["product_id", "product_name", "aisle_id", "department_id"],
        dtype={"product_id": "int32", "aisle_id": "int16", "department_id": "int16"},
    )
    dims["p_is_organic"] = dims["product_name"].str.contains(
        "organic", case=False, na=False
    ).astype("int8")
    dims = dims.drop(columns="product_name").drop_duplicates("product_id")

    raw_target_sizes = target_lines.groupby("user_id")["product_id"].nunique().to_dict()
    raw_target_items = int(target_lines["product_id"].nunique())
    prior_lines = prior_lines.loc[prior_lines["product_id"].isin(allowed)].copy()
    target_lines = target_lines.loc[target_lines["product_id"].isin(allowed)].copy()

    # Drop only users with no target item in the shared vocabulary or no mapped history.
    target_users_mapped = set(target_lines["user_id"].astype(int))
    history_users_mapped = set(prior_lines["user_id"].astype(int))
    final_users = selected & target_users_mapped & history_users_mapped
    if len(final_users) < 10:
        raise ValueError("Fewer than 10 users remain after shared-vocabulary filtering.")
    prior_meta = prior_meta.loc[prior_meta["user_id"].isin(final_users)].copy()
    prior_lines = prior_lines.loc[prior_lines["user_id"].isin(final_users)].copy()
    target_meta = target_meta.loc[target_meta["user_id"].isin(final_users)].copy()
    target_lines = target_lines.loc[target_lines["user_id"].isin(final_users)].copy()
    split_map = {u: s for u, s in split_map.items() if u in final_users}
    splits = {s: [u for u in ids if u in final_users] for s, ids in splits.items()}

    # All final split groups must be represented.
    if any(len(splits[s]) == 0 for s in ("train", "valid", "test")):
        raise ValueError("A split became empty after vocabulary filtering. Increase max_users/top_items.")

    # Store target basket size and mapping coverage for each user.
    mapped_target_sizes = target_lines.groupby("user_id")["product_id"].nunique().to_dict()
    raw_target_sizes = {int(k): int(v) for k, v in raw_target_sizes.items() if int(k) in final_users}
    vocab_coverage = (
        sum(mapped_target_sizes.values()) / max(1, sum(raw_target_sizes.values()))
    )

    # User/order/target mapping metadata for later feature calculation.
    # Both item tables already carry user/order metadata from the validated
    # many-to-one joins above; keep one copy rather than performing a second merge.
    history_lines = prior_lines.copy()
    target_lines = target_lines.copy()

    # Persist all shared artefacts. This exact split/mapping is reused for every H.
    pd.DataFrame({"user_id": sorted(final_users),
                  "split": [split_map[u] for u in sorted(final_users)]}).to_csv(
        out_dir / "user_splits.csv", index=False
    )
    pd.DataFrame({"original_product_id": list(item_map.keys()),
                  "model_product_id": list(item_map.values())}).to_csv(
        out_dir / "item_mapping.csv", index=False
    )
    pd.DataFrame({"user_id": list(mapped_target_sizes),
                  "mapped_target_size": list(mapped_target_sizes.values()),
                  "raw_target_size": [raw_target_sizes.get(u, 0) for u in mapped_target_sizes]}).to_csv(
        out_dir / "target_sizes.csv", index=False
    )
    history_lines.to_csv(out_dir / "history_lines_maxH.csv.gz", index=False, compression="gzip")
    target_lines.to_csv(out_dir / "target_lines.csv.gz", index=False, compression="gzip")
    prior_meta.to_csv(out_dir / "history_orders_maxH.csv.gz", index=False, compression="gzip")
    dims.to_csv(out_dir / "product_dimensions.csv", index=False)

    item_map_df = pd.DataFrame({"original_product_id": list(item_map.keys()),
                                "model_product_id": list(item_map.values())})
    history_lengths = hs
    _write_sets2sets_files(
        out_dir, prior_meta, history_lines, target_lines, item_map,
        sorted(final_users), history_lengths
    )

    summary = {
        "dataset": "Instacart Market Basket Analysis",
        "target_definition": "One eval_set=train basket per user; official eval_set=test labels are hidden",
        "history_lengths": hs,
        "max_history": max_h,
        "users_selected_initial": len(selected),
        "users_final": len(final_users),
        "train_users": len(splits["train"]),
        "valid_users": len(splits["valid"]),
        "test_users": len(splits["test"]),
        "item_vocabulary_size": len(item_map),
        "target_item_vocabulary_coverage": float(vocab_coverage),
        "split_ratio_requested": {"train": 0.72, "valid": 0.08, "test": 0.20},
        "seed": seed,
        "max_users_requested": max_users,
        "top_items": top_items,
        "important_note": "The item vocabulary is built from TRAIN users' prior history only. Target labels are not used in features or global popularity.",
    }
    (out_dir / "preparation_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print("Preparation complete:")
    print(json.dumps(summary, indent=2))

    return {
        "history_meta": prior_meta.reset_index(drop=True),
        "history_lines": history_lines.reset_index(drop=True),
        "target_meta": target_meta.reset_index(drop=True),
        "target_lines": target_lines.reset_index(drop=True),
        "product_dims": dims.reset_index(drop=True),
        "split_map": split_map,
        "splits": splits,
        "item_map": item_map,
        "item_mapping": item_map_df,
        "mapped_target_sizes": {int(k): int(v) for k, v in mapped_target_sizes.items()},
        "raw_target_sizes": raw_target_sizes,
        "summary": summary,
        "out_dir": out_dir,
    }


def _build_user_and_category_stats(
    meta_h: pd.DataFrame,
    hist_h: pd.DataFrame,
    dims: pd.DataFrame,
    history_length: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Compute user, user-product, user-aisle and user-department statistics."""
    meta_agg = meta_h.groupby("user_id").agg(
        u_hist_baskets=("order_id", "nunique"),
        u_avg_days_gap=("days_since_prior_order", "mean"),
        u_std_days_gap=("days_since_prior_order", "std"),
        u_dow_std=("order_dow", "std"),
        u_hour_std=("order_hour_of_day", "std"),
    ).reset_index()
    basket_sizes = hist_h.groupby(["user_id", "order_id"]).size().rename("basket_size").reset_index()
    basket_agg = basket_sizes.groupby("user_id").agg(
        u_avg_basket_size=("basket_size", "mean"),
        u_basket_size_std=("basket_size", "std"),
    ).reset_index()
    user_item_agg = hist_h.groupby("user_id").agg(
        u_hist_item_count=("product_id", "size"),
        u_unique_products=("product_id", "nunique"),
        u_reorder_rate=("h_reordered", "mean"),
    ).reset_index()
    user_stats = meta_agg.merge(basket_agg, on="user_id", how="left").merge(
        user_item_agg, on="user_id", how="left"
    )
    user_stats["u_basket_size_std"] = user_stats["u_basket_size_std"].fillna(0)
    user_stats["u_std_days_gap"] = user_stats["u_std_days_gap"].fillna(0)
    user_stats["u_dow_std"] = user_stats["u_dow_std"].fillna(0)
    user_stats["u_hour_std"] = user_stats["u_hour_std"].fillna(0)

    up = hist_h.groupby(["user_id", "product_id"]).agg(
        up_purchase_count=("order_id", "nunique"),
        up_reorder_rate=("h_reordered", "mean"),
        up_last_seen_age=("age_from_latest", "min"),
        up_last3_count=("age_from_latest", lambda x: int((x < 3).sum())),
        up_avg_cart_position=("add_to_cart_order", "mean"),
        up_recency_weighted_count=("recency_weight", "sum"),
    ).reset_index()
    up["up_basket_frequency"] = up["up_purchase_count"] / float(history_length)
    up["up_recency_score"] = 1.0 / (1.0 + up["up_last_seen_age"])
    up["up_in_last_basket"] = (up["up_last_seen_age"] == 0).astype("int8")

    with_dims = hist_h.merge(
        dims[["product_id", "aisle_id", "department_id"]],
        on="product_id", how="left", validate="many_to_one"
    )
    user_totals = with_dims.groupby("user_id").size().rename("u_cat_total")
    ua = with_dims.groupby(["user_id", "aisle_id"]).size().rename("ua_purchase_count").reset_index()
    ua = ua.merge(user_totals, on="user_id", how="left")
    ua["ua_purchase_share"] = ua["ua_purchase_count"] / ua["u_cat_total"].clip(lower=1)
    ud = with_dims.groupby(["user_id", "department_id"]).size().rename("ud_purchase_count").reset_index()
    ud = ud.merge(user_totals, on="user_id", how="left")
    ud["ud_purchase_share"] = ud["ud_purchase_count"] / ud["u_cat_total"].clip(lower=1)
    return user_stats, up, ua.drop(columns="u_cat_total"), ud.drop(columns="u_cat_total")


def _ranking_metrics(frame: pd.DataFrame, score_col: str, ks: Iterable[int]) -> dict[str, float]:
    """Compute ranking metrics against the complete mapped target basket per user."""
    ordered = frame.sort_values(
        ["user_id", score_col, "product_id"],
        ascending=[True, False, True], kind="mergesort"
    )
    results: dict[str, list[float]] = {f"{metric}@{k}": [] for k in ks
        for metric in ("Precision", "Recall", "NDCG", "HitRate", "MAP")}
    for _, group in ordered.groupby("user_id", sort=False):
        rel = group["label"].to_numpy(dtype=np.int8)
        target_size = max(1, int(group["target_basket_size"].iloc[0]))
        for k in ks:
            top = rel[:k]
            hits = int(top.sum())
            denom_precision = max(1, len(top))
            precision = hits / denom_precision
            recall = hits / target_size
            discounts = 1.0 / np.log2(np.arange(2, len(top) + 2))
            dcg = float((top * discounts).sum())
            ideal_n = min(target_size, k)
            idcg = float((1.0 / np.log2(np.arange(2, ideal_n + 2))).sum()) if ideal_n else 0.0
            ndcg = dcg / idcg if idcg > 0 else 0.0
            hit_rate = float(hits > 0)
            cumulative_hits = np.cumsum(top)
            precisions = cumulative_hits / np.arange(1, len(top) + 1)
            ap = float((precisions * top).sum() / max(1, min(target_size, k)))
            results[f"Precision@{k}"].append(precision)
            results[f"Recall@{k}"].append(recall)
            results[f"NDCG@{k}"].append(ndcg)
            results[f"HitRate@{k}"].append(hit_rate)
            results[f"MAP@{k}"].append(ap)
    return {key: float(np.mean(values)) if values else 0.0 for key, values in results.items()}


def build_candidate_frame(
    bundle: dict,
    history_length: int,
    global_top_n: int = 200,
) -> tuple[pd.DataFrame, dict]:
    """Build history ∪ train-history popularity candidates and XCAI2-style features."""
    if history_length < 1 or history_length > int(bundle["summary"]["max_history"]):
        raise ValueError(
            f"H={history_length} is outside the prepared history range "
            f"1..{bundle['summary']['max_history']}."
        )
    h = int(history_length)
    history_meta = bundle["history_meta"].sort_values(["user_id", "order_number"])
    meta_h = history_meta.groupby("user_id", sort=False, group_keys=False).tail(h).copy()
    order_ids_h = set(meta_h["order_id"].astype(int))
    hist_h = bundle["history_lines"].loc[
        bundle["history_lines"]["order_id"].isin(order_ids_h)
    ].copy()
    if hist_h.empty:
        raise ValueError(f"H={h} has no history item rows.")

    meta_order = meta_h.sort_values(["user_id", "order_number"]).copy()
    meta_order["basket_rank"] = meta_order.groupby("user_id").cumcount() + 1
    meta_order["age_from_latest"] = h - meta_order["basket_rank"]
    hist_h = hist_h.merge(
        meta_order[["order_id", "age_from_latest"]], on="order_id", how="left",
        validate="many_to_one"
    )
    hist_h = hist_h.sort_values(["user_id", "product_id", "order_number"])
    hist_h["h_reordered"] = (
        hist_h.groupby(["user_id", "product_id"]).cumcount() > 0
    ).astype("int8")
    hist_h["recency_weight"] = 1.0 / (1.0 + hist_h["age_from_latest"].astype(float))

    train_users = set(bundle["splits"]["train"])
    train_hist = hist_h.loc[hist_h["user_id"].isin(train_users)].copy()
    if train_hist.empty:
        raise ValueError("No TRAIN history rows for global product features.")
    pop_counts = train_hist["product_id"].value_counts()
    popular_items = pop_counts.head(int(global_top_n)).index.astype(int).tolist()
    pop_set = set(popular_items)

    # All global product/category feature tables use TRAIN users' prior history only.
    p_stats = train_hist.groupby("product_id").agg(
        p_total_purchases_train=("order_id", "size"),
        p_distinct_users_train=("user_id", "nunique"),
        p_reorder_rate_train=("h_reordered", "mean"),
        p_avg_cart_position_train=("add_to_cart_order", "mean"),
    ).reset_index()
    p_stats["p_global_pop_score"] = p_stats["p_total_purchases_train"] / max(1, int(pop_counts.max()))
    rank_map = {int(pid): rank + 1 for rank, pid in enumerate(pop_counts.index.tolist())}
    p_stats["p_global_pop_rank"] = p_stats["product_id"].map(rank_map).fillna(len(rank_map) + 1)

    train_with_dims = train_hist.merge(
        bundle["product_dims"][["product_id", "aisle_id", "department_id", "p_is_organic"]],
        on="product_id", how="left", validate="many_to_one"
    )
    a_stats = train_with_dims.groupby("aisle_id").agg(
        a_total_purchases_train=("product_id", "size"),
        a_reorder_rate_train=("h_reordered", "mean"),
    ).reset_index()
    d_stats = train_with_dims.groupby("department_id").agg(
        d_total_purchases_train=("product_id", "size"),
        d_reorder_rate_train=("h_reordered", "mean"),
    ).reset_index()

    user_stats, up_stats, ua_stats, ud_stats = _build_user_and_category_stats(
        meta_h, hist_h, bundle["product_dims"], h
    )

    # Retrieval candidates. Training target positives are appended only to form supervised
    # positive examples; they are never used to compute features or exposed via flags.
    history_candidates = hist_h[["user_id", "product_id"]].drop_duplicates()
    all_users = sorted(bundle["split_map"])
    pop_frame = pd.DataFrame({"product_id": popular_items})
    pop_frame["_key"] = 1
    users_frame = pd.DataFrame({"user_id": all_users, "_key": 1})
    pop_candidates = users_frame.merge(pop_frame, on="_key", how="inner").drop(columns="_key")
    pairs = pd.concat([history_candidates, pop_candidates], ignore_index=True).drop_duplicates(
        ["user_id", "product_id"]
    )
    train_target_pairs = bundle["target_lines"].loc[
        bundle["target_lines"]["user_id"].isin(train_users), ["user_id", "product_id"]
    ].drop_duplicates()
    pairs = pd.concat([pairs, train_target_pairs], ignore_index=True).drop_duplicates(
        ["user_id", "product_id"]
    )

    # Candidate-source flags are kept only for audit/export, never in FEATURE_COLUMNS.
    pairs = pairs.merge(
        history_candidates.assign(is_history_candidate=1),
        on=["user_id", "product_id"], how="left", validate="one_to_one"
    )
    pop_item_flags = pop_candidates[["product_id"]].drop_duplicates().assign(
        is_global_pop_candidate=1
    )
    pairs = pairs.merge(
        pop_item_flags,
        on="product_id", how="left", validate="many_to_one"
    )
    pairs[["is_history_candidate", "is_global_pop_candidate"]] = pairs[
        ["is_history_candidate", "is_global_pop_candidate"]
    ].fillna(0).astype("int8")

    # Labels and the full mapped target-basket size. This preserves missed target items
    # in the recall denominator instead of silently evaluating only retrieved positives.
    target_pairs = bundle["target_lines"][["user_id", "product_id"]].drop_duplicates().assign(label=1)
    pairs = pairs.merge(target_pairs, on=["user_id", "product_id"], how="left", validate="one_to_one")
    pairs["label"] = pairs["label"].fillna(0).astype("int8")
    target_size_map = bundle["mapped_target_sizes"]
    pairs["target_basket_size"] = pairs["user_id"].map(target_size_map).fillna(0).astype("int16")
    pairs["split"] = pairs["user_id"].map(bundle["split_map"])

    # Attach user, user-product, product and category features.
    pairs = pairs.merge(user_stats, on="user_id", how="left", validate="many_to_one")
    pairs = pairs.merge(up_stats, on=["user_id", "product_id"], how="left", validate="one_to_one")
    pairs = pairs.merge(p_stats, on="product_id", how="left", validate="many_to_one")
    pairs = pairs.merge(
        bundle["product_dims"][["product_id", "aisle_id", "department_id", "p_is_organic"]],
        on="product_id", how="left", validate="many_to_one", suffixes=("", "_dim")
    )
    pairs = pairs.merge(a_stats, on="aisle_id", how="left", validate="many_to_one")
    pairs = pairs.merge(d_stats, on="department_id", how="left", validate="many_to_one")
    pairs = pairs.merge(ua_stats, on=["user_id", "aisle_id"], how="left", validate="many_to_one")
    pairs = pairs.merge(ud_stats, on=["user_id", "department_id"], how="left", validate="many_to_one")
    pairs["p_is_organic"] = pairs["p_is_organic"].fillna(0)
    for col in FEATURE_COLUMNS:
        if col not in pairs:
            pairs[col] = 0.0
        pairs[col] = pd.to_numeric(pairs[col], errors="coerce").fillna(0).astype("float32")
    pairs["history_frequency_score"] = pairs["up_basket_frequency"].astype("float32")
    pairs["recency_weighted_score"] = pairs["up_recency_weighted_count"].astype("float32")
    pairs["global_popularity_score"] = pairs["p_global_pop_score"].astype("float32")
    pairs["candidate_coverage_user"] = pairs.groupby("user_id")["label"].transform("sum") / pairs[
        "target_basket_size"
    ].clip(lower=1)

    # Avoid silently using target labels to influence any feature columns.
    if any(c in FEATURE_COLUMNS for c in ["label", "target_basket_size", "is_history_candidate", "is_global_pop_candidate"]):
        raise AssertionError("Leakage field found in model feature list.")
    info = {
        "history_length": h,
        "global_popular_items": len(popular_items),
        "candidate_pairs": int(len(pairs)),
        "candidate_users": int(pairs["user_id"].nunique()),
        "train_positive_candidates_added": int(len(train_target_pairs)),
        "feature_columns": FEATURE_COLUMNS.copy(),
    }
    return pairs, info


def _compute_test_coverage(test_frame: pd.DataFrame) -> dict[str, float]:
    per_user = test_frame.groupby("user_id").agg(
        retrieved_target_items=("label", "sum"),
        target_items=("target_basket_size", "first"),
        candidates=("product_id", "nunique"),
        candidate_coverage=("candidate_coverage_user", "first"),
    )
    total_retrieved = int(per_user["retrieved_target_items"].sum())
    total_target = int(per_user["target_items"].sum())
    return {
        "candidate_coverage_macro": float(per_user["candidate_coverage"].mean()),
        "candidate_coverage_micro": float(total_retrieved / max(1, total_target)),
        "target_items_test_mapped": total_target,
        "target_items_test_retrieved": total_retrieved,
        "mean_candidates_per_user": float(per_user["candidates"].mean()),
    }


def _fit_lightgbm(train: pd.DataFrame, valid: pd.DataFrame, seed: int = 42):
    """Fit LightGBM with validation early stopping; preserve import error details."""
    try:
        from lightgbm import LGBMClassifier, early_stopping, log_evaluation
    except Exception as exc:  # show the actual ABI/dependency error, not a false 'not installed'
        raise RuntimeError(
            "LightGBM could not be imported. This often indicates an incompatible binary "
            "installation. Restart the runtime after installing the pinned requirements. "
            f"Original import error: {type(exc).__name__}: {exc}"
        ) from exc

    positives = int(train["label"].sum())
    negatives = int(len(train) - positives)
    if positives == 0 or negatives == 0:
        raise ValueError("Training data must contain both positive and negative candidate rows.")
    scale = min(30.0, negatives / max(1, positives))
    params = dict(
        objective="binary", n_estimators=700, learning_rate=0.04,
        num_leaves=31, max_depth=-1, min_child_samples=30,
        subsample=0.85, subsample_freq=1, colsample_bytree=0.85,
        reg_lambda=2.0, scale_pos_weight=scale,
        random_state=seed, n_jobs=-1, verbosity=-1,
    )
    model = LGBMClassifier(**params)
    callbacks = [early_stopping(50, verbose=False), log_evaluation(0)]
    has_valid_classes = valid["label"].nunique() == 2
    if has_valid_classes and len(valid):
        model.fit(
            train[FEATURE_COLUMNS], train["label"],
            eval_set=[(valid[FEATURE_COLUMNS], valid["label"])],
            eval_metric="auc", callbacks=callbacks,
        )
        best_iter = int(model.best_iteration_ or params["n_estimators"])
    else:
        model.fit(train[FEATURE_COLUMNS], train["label"])
        best_iter = int(params["n_estimators"])

    combined = pd.concat([train, valid], ignore_index=True)
    pos2 = int(combined["label"].sum())
    neg2 = int(len(combined) - pos2)
    params["n_estimators"] = best_iter
    params["scale_pos_weight"] = min(30.0, neg2 / max(1, pos2))
    final_model = LGBMClassifier(**params)
    final_model.fit(combined[FEATURE_COLUMNS], combined["label"])
    return final_model, best_iter


def run_experiments(
    bundle: dict,
    output_dir: str | Path,
    history_lengths: Iterable[int] = (4, 8, 13),
    global_top_n: int = 200,
    ks: Iterable[int] = (5, 10, 20),
    seed: int = 42,
) -> pd.DataFrame:
    """Run history-frequency, global-popularity, recency and LightGBM models."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    hs = [int(h) for h in history_lengths]
    ks = tuple(int(k) for k in ks)
    if not hs or any(h < 1 for h in hs) or not ks or any(k < 1 for k in ks):
        raise ValueError("history_lengths and ks must contain positive integers.")
    if global_top_n < max(ks):
        raise ValueError("global_top_n must be >= max(K).")
    result_rows = []
    all_test_predictions = []

    for h in hs:
        print("\n" + "=" * 78)
        print(f"XCAI2-inspired feature candidate model | H={h}")
        print("=" * 78)
        t0 = time.perf_counter()
        frame, info = build_candidate_frame(bundle, h, global_top_n)
        frame["split"] = frame["split"].astype(str)
        train = frame.loc[frame["split"].eq("train")].copy()
        valid = frame.loc[frame["split"].eq("valid")].copy()
        test = frame.loc[frame["split"].eq("test")].copy()
        if train.empty or valid.empty or test.empty:
            raise ValueError("One of train/validation/test candidate groups is empty.")
        if train["label"].nunique() < 2:
            raise ValueError("Training candidate table has only one label class.")

        # Export candidate pool for later model-only comparison with Sets2Sets.
        pool_cols = ["user_id", "product_id", "split", "label", "target_basket_size",
                     "is_history_candidate", "is_global_pop_candidate", "candidate_coverage_user"]
        frame[pool_cols].to_csv(
            output_dir / f"candidate_pool_H{h}.csv.gz", index=False, compression="gzip"
        )

        coverage = _compute_test_coverage(test)
        baseline_specs = [
            ("UserHistoryFrequency", "history_frequency_score"),
            ("RecencyWeightedHistory", "recency_weighted_score"),
            ("GlobalPopularity", "global_popularity_score"),
        ]
        for name, score_col in baseline_specs:
            metrics = _ranking_metrics(test, score_col, ks)
            result_rows.append({"H": h, "Model": name, **metrics, **coverage})

        print(f"Candidate users: {info['candidate_users']:,}; candidate rows: {info['candidate_pairs']:,}")
        print(f"Test candidate coverage (macro): {coverage['candidate_coverage_macro']:.4f}")
        model, best_iter = _fit_lightgbm(train, valid, seed)
        test_scored = test.copy()
        test_scored["lightgbm_score"] = model.predict_proba(
            test_scored[FEATURE_COLUMNS]
        )[:, 1].astype("float32")
        metrics = _ranking_metrics(test_scored, "lightgbm_score", ks)
        elapsed = time.perf_counter() - t0
        result_rows.append({
            "H": h, "Model": "XCAI2_LightGBM", **metrics, **coverage,
            "best_iteration": best_iter, "elapsed_seconds": round(elapsed, 2),
        })

        keep = ["user_id", "product_id", "label", "target_basket_size",
                "lightgbm_score", "history_frequency_score", "recency_weighted_score",
                "global_popularity_score", "candidate_coverage_user"]
        test_scored[keep].to_csv(
            output_dir / f"test_rankings_H{h}.csv.gz", index=False, compression="gzip"
        )
        importance = pd.DataFrame({
            "feature": FEATURE_COLUMNS,
            "importance_gain": model.booster_.feature_importance(importance_type="gain"),
            "importance_split": model.booster_.feature_importance(importance_type="split"),
        }).sort_values("importance_gain", ascending=False)
        importance.to_csv(output_dir / f"feature_importance_H{h}.csv", index=False)
        all_test_predictions.append(test_scored[keep].assign(H=h))
        print(f"LightGBM best iteration: {best_iter}; elapsed: {elapsed:.1f}s")
        print(pd.DataFrame([{"H": h, "Model": "XCAI2_LightGBM", **metrics}]).to_string(index=False))

    results = pd.DataFrame(result_rows)
    results.to_csv(output_dir / "experiment_results.csv", index=False)
    if all_test_predictions:
        pd.concat(all_test_predictions, ignore_index=True).to_csv(
            output_dir / "all_test_rankings.csv.gz", index=False, compression="gzip"
        )
    metadata = {
        "dataset": "Instacart Market Basket Analysis",
        "research_task": "Predict one labeled next basket from the latest H prior baskets",
        "history_lengths": hs,
        "global_top_n": global_top_n,
        "ks": list(ks),
        "split": "fixed user-level 72/8/20; same users across all H",
        "global_features": "computed only from TRAIN users' H-history prior baskets",
        "target_usage": "target items are used for labels; training positives are added as supervised rows; never used as model features or global popularity",
        "model": "XCAI2-inspired user-product feature scorer using LightGBM",
        "metrics_note": "Recall/NDCG/MAP denominators use the complete mapped target basket; candidate coverage is reported separately.",
        "seed": seed,
    }
    (output_dir / "experiment_metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )
    print("\nSaved results:", output_dir / "experiment_results.csv")
    return results


## Cell 9 — Import source và kiểm tra

In [ ]:
import sys
sys.path.insert(0, str(LOCAL_PROJECT))

from src.nbr_pipeline import (
    verify_dataset,
    prepare_cohort,
    build_candidate_frame,
    run_experiments,
    FEATURE_COLUMNS,
)

verify_dataset(DATA_DIR)
print("Pipeline imported successfully.")
print("Number of model features:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

## Cell 10 — Chuẩn bị cohort dùng chung cho H=4/8/13

Bước này đọc `orders.csv`, chọn user có target được gán nhãn và có ít nhất 13 prior baskets, sau đó chỉ quét các item rows thuộc 13 prior orders mới nhất của cohort. File prior lớn được xử lý theo chunks để tránh nạp toàn bộ 32 triệu dòng vào RAM.

Kết quả gồm một user split cố định, một item mapping chung, target basket và các file `history/future` cho Sets2Sets.

In [ ]:
bundle = prepare_cohort(
    data_dir=DATA_DIR,
    out_dir=PREPARED_DIR,
    history_lengths=HISTORY_LENGTHS,
    max_users=MAX_USERS,
    top_items=TOP_ITEMS,
    seed=SEED,
    chunksize=CHUNK_SIZE,
)

print("\nCohort preparation finished.")

## Cell 11 — Kiểm tra cohort, target coverage và file Sets2Sets

In [ ]:
prep_summary = json.loads((PREPARED_DIR / 'preparation_summary.json').read_text())
print(json.dumps(prep_summary, indent=2))

user_splits = pd.read_csv(PREPARED_DIR / 'user_splits.csv')
item_mapping = pd.read_csv(PREPARED_DIR / 'item_mapping.csv')
print("\nUser split counts:")
display(user_splits['split'].value_counts().rename_axis('split').to_frame('users'))
print("Item vocabulary size:", len(item_mapping))

prepared_files = sorted(p.name for p in PREPARED_DIR.glob('*'))
print("\nPrepared files:")
for name in prepared_files:
    print(" -", name)

assert set(user_splits['split']) == {'train', 'valid', 'test'}
assert len(item_mapping) > 0
assert prep_summary['target_item_vocabulary_coverage'] > 0
print("\nPrepared-data checks: PASSED")

## Cell 12 — Chạy mô hình và baseline với H=4

H=4 được chạy riêng để nếu có lỗi, ta chỉ cần chẩn đoán một thí nghiệm. Output được đặt trong thư mục riêng, không ghi đè H=8 hoặc H=13.

In [ ]:
RESULTS_H4 = RESULTS_ROOT / 'H4'
results_h4 = run_experiments(
    bundle=bundle,
    output_dir=RESULTS_H4,
    history_lengths=[4],
    global_top_n=GLOBAL_TOP_N,
    ks=K_LIST,
    seed=SEED,
)
display(results_h4)

## Cell 13 — Chạy H=8

In [ ]:
RESULTS_H8 = RESULTS_ROOT / 'H8'
results_h8 = run_experiments(
    bundle=bundle,
    output_dir=RESULTS_H8,
    history_lengths=[8],
    global_top_n=GLOBAL_TOP_N,
    ks=K_LIST,
    seed=SEED,
)
display(results_h8)

## Cell 14 — Chạy H=13

In [ ]:
RESULTS_H13 = RESULTS_ROOT / 'H13'
results_h13 = run_experiments(
    bundle=bundle,
    output_dir=RESULTS_H13,
    history_lengths=[13],
    global_top_n=GLOBAL_TOP_N,
    ks=K_LIST,
    seed=SEED,
)
display(results_h13)

## Cell 15 — Gộp kết quả của H=4/8/13

In [ ]:
result_files = [
    RESULTS_H4 / 'experiment_results.csv',
    RESULTS_H8 / 'experiment_results.csv',
    RESULTS_H13 / 'experiment_results.csv',
]
missing = [str(p) for p in result_files if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Missing result files: {missing}. Run all H cells first.")

results_all = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
results_all = results_all.sort_values(['H', 'Model']).reset_index(drop=True)
combined_path = RESULTS_ROOT / 'experiment_results_all_H.csv'
results_all.to_csv(combined_path, index=False)

display(results_all)
print("Saved:", combined_path)

## Cell 16 — So sánh Recall@K và NDCG@K theo H

In [ ]:
import matplotlib.pyplot as plt

model_order = [
    'UserHistoryFrequency',
    'RecencyWeightedHistory',
    'GlobalPopularity',
    'XCAI2_LightGBM',
]
plot_df = results_all[results_all['Model'].isin(model_order)].copy()

for metric in ['Recall@10', 'NDCG@10', 'HitRate@10']:
    if metric not in plot_df.columns:
        continue
    pivot = plot_df.pivot(index='H', columns='Model', values=metric)
    ax = pivot.reindex(columns=[m for m in model_order if m in pivot.columns]).plot(
        marker='o', figsize=(9, 5)
    )
    ax.set_title(f'{metric} across history windows')
    ax.set_xlabel('History length (H)')
    ax.set_ylabel(metric)
    ax.set_xticks(HISTORY_LENGTHS)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

## Cell 17 — Kiểm tra candidate coverage

In [ ]:
coverage_cols = [
    'H', 'Model', 'candidate_coverage_macro', 'candidate_coverage_micro',
    'target_items_test_mapped', 'target_items_test_retrieved',
    'mean_candidates_per_user'
]
coverage_table = results_all[coverage_cols].drop_duplicates().sort_values(['H', 'Model'])
display(coverage_table)

print("Lưu ý: candidate coverage đo tỷ lệ target items được retrieval đưa vào candidate pool; đây là giới hạn trên của ranking model.")

## Cell 18 — Feature importance của LightGBM

In [ ]:
importance_frames = []
for h, folder in [(4, RESULTS_H4), (8, RESULTS_H8), (13, RESULTS_H13)]:
    path = folder / f'feature_importance_H{h}.csv'
    imp = pd.read_csv(path)
    imp['H'] = h
    importance_frames.append(imp)

importance_all = pd.concat(importance_frames, ignore_index=True)
for h in HISTORY_LENGTHS:
    display(importance_all[importance_all['H'] == h].head(15))

importance_all.to_csv(RESULTS_ROOT / 'feature_importance_all_H.csv', index=False)

## Cell 19 — Case study: đề xuất Top-10 cho một user test

Chọn một test user cố định, xem đề xuất H=13 và đối chiếu với target basket có nhãn trong offline evaluation.

In [ ]:
test_user = int(user_splits.loc[user_splits['split'].eq('test'), 'user_id'].iloc[0])
rank_path = RESULTS_H13 / 'test_rankings_H13.csv.gz'
rank_df = pd.read_csv(rank_path)
products_df = pd.read_csv(DATA_DIR / 'products.csv', usecols=['product_id', 'product_name'])

user_recs = rank_df.loc[rank_df['user_id'].eq(test_user)].sort_values(
    'lightgbm_score', ascending=False
).head(10).merge(products_df, on='product_id', how='left')
user_recs['is_in_target_basket'] = user_recs['label'].astype(bool)
print("Test user:", test_user)
display(user_recs[['product_id', 'product_name', 'lightgbm_score', 'is_in_target_basket']])

## Cell 20 — Kiểm tra file đầu vào cho Sets2Sets

In [ ]:
sets2sets_files = []
for h in HISTORY_LENGTHS:
    for suffix in ['history', 'future']:
        p = PREPARED_DIR / f'instacart_H{h}_{suffix}.csv'
        if not p.is_file():
            raise FileNotFoundError(p)
        sets2sets_files.append((p.name, p.stat().st_size / 1024**2))

display(pd.DataFrame(sets2sets_files, columns=['file', 'size_MiB']))
print("\nShared split:", PREPARED_DIR / 'user_splits.csv')
print("Shared item mapping:", PREPARED_DIR / 'item_mapping.csv')
print("\nDùng đúng PREPARED_DIR này trong runner Sets2Sets để giữ cùng cohort/split/vocabulary.")

## Cell 21 — Lưu cấu hình và kết luận thí nghiệm

Kết quả smoke test chỉ dùng để xác nhận pipeline. Không dùng 1.000 users làm kết luận cuối cùng nếu mục tiêu là báo cáo nghiên cứu.

In [ ]:
run_metadata = {
    'dataset': 'Instacart Market Basket Analysis',
    'dataset_source': 'KaggleHub mirror: psparks/instacart-market-basket-analysis',
    'history_lengths': HISTORY_LENGTHS,
    'max_users': MAX_USERS,
    'top_items': TOP_ITEMS,
    'global_top_n': GLOBAL_TOP_N,
    'ks': K_LIST,
    'seed': SEED,
    'prepared_dir': str(PREPARED_DIR),
    'results_root': str(RESULTS_ROOT),
    'python': sys.version,
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'lightgbm': lightgbm.__version__,
    'target_definition': "One labeled eval_set='train' basket per user",
    'caution': 'Smoke-test results; increase cohort and repeat with a new RUN_TAG for final analysis.'
}
(RESULTS_ROOT / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2))
print(json.dumps(run_metadata, indent=2))
print("\nCOMPLETE: Instacart XCAI2-inspired Next-Basket experiment pipeline finished.")

## Kế hoạch thực nghiệm tiếp theo

Sau khi smoke test hoàn thành:

1. Đổi `MAX_USERS = 5000` và `RUN_TAG` tương ứng, chạy lại notebook từ Cell 6 để có một cohort mới.
2. Nếu Colab RAM/CPU đáp ứng, thực hiện trên toàn bộ user đủ điều kiện (`MAX_USERS = None`).
3. Chạy Sets2Sets bằng **cùng** `PREPARED_DIR/user_splits.csv`, `item_mapping.csv` và các file history/future tương ứng.
4. So sánh model + retrieval pipeline và báo cáo candidate coverage; nếu muốn so sánh riêng model, lọc Sets2Sets scores về đúng `candidate_pool_H{H}.csv.gz`.
5. Báo cáo độ biến thiên nếu có thể bằng nhiều seed hoặc nhiều user split. Một lần split duy nhất chỉ nên được xem là một thí nghiệm offline.

**Không kết luận H nào tốt nhất chỉ từ smoke test.**